# Signal review
Loads saved raw/filtered Recording definitions. Reads short windows only; never sorts or materializes a complete recording. Run 01_preprocessing.py first. All parameter alternatives below are exploratory.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'config.yaml').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from utils import load_config, read_json
config = load_config(ROOT / 'config.yaml')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import spikeinterface as si
import spikeinterface.preprocessing as sp
from scipy.signal import welch, sosfreqz
folder = Path(config['paths']['preprocessing'])
recording = si.load(folder / 'metadata/raw_recording.json')
recording_filtered = si.load(folder / 'metadata/filtered_recording.json')
display(pd.read_csv(folder / 'metadata/channel_map.csv'))
display(pd.read_csv(folder / 'metrics/signal_qc.csv'))
print(read_json(folder / 'metadata/recording.json'))


In [ ]:
# Edit a bounded window and channel subset, in seconds within this one recording.
fs = recording.get_sampling_frequency()
start_sec, window_sec = 0., .5
channel_ids = recording.get_channel_ids()[:4]
first, last = round(start_sec*fs), min(recording.get_num_samples(), round((start_sec+window_sec)*fs))
raw = recording.get_traces(start_frame=first, end_frame=last, channel_ids=channel_ids)
raw_uV = recording.get_traces(start_frame=first, end_frame=last, channel_ids=channel_ids, return_in_uV=True)
filtered_uV = recording_filtered.get_traces(start_frame=first, end_frame=last, channel_ids=channel_ids, return_in_uV=True)
gains = recording.get_channel_gains()[:len(channel_ids)]
offsets = recording.get_channel_offsets()[:len(channel_ids)]
print('Raw scaling max error (uV):', np.max(np.abs(raw_uV-(raw.astype('float32')*gains+offsets))))
fig, axes = plt.subplots(len(channel_ids), 2, figsize=(12, 2*len(channel_ids)), squeeze=False)
for c, channel in enumerate(channel_ids):
    axes[c, 0].plot((np.arange(len(raw))+first)/fs, raw_uV[:, c])
    axes[c, 1].plot((np.arange(len(raw))+first)/fs, filtered_uV[:, c])
    for ax in axes[c]:
        ax.set(xlabel='Time (s)', ylabel='uV', title=str(channel))
plt.tight_layout()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for label, values in [('raw', raw_uV), ('filtered', filtered_uV)]:
    f, psd = welch(values[:, 0], fs=fs, nperseg=min(2048, len(values)))
    axes[0].semilogy(f, psd, label=label)
coefficients = np.asarray(read_json(folder / 'metadata/filter_parameters.json')['coefficients'])
f, response = sosfreqz(coefficients, worN=4096, fs=fs)
# Forward/backward filtering has squared single-pass magnitude response.
axes[1].plot(f, np.abs(response)**2)
axes[0].set(xlabel='Hz', ylabel='PSD (uV^2/Hz)')
axes[0].legend()
axes[1].set(xlabel='Hz', ylabel='Forward/backward magnitude', title='Filter response')
plt.tight_layout()


In [ ]:
# Exploratory alternative; changes neither config nor saved production Recording.
alternative_parameters = {**config['preprocessing'], 'freq_min': 500}
alternative = sp.bandpass_filter(recording, **alternative_parameters)
alternative_uV = alternative.get_traces(start_frame=first, end_frame=last, channel_ids=channel_ids, return_in_uV=True)
plt.plot(np.arange(len(filtered_uV))/fs, filtered_uV[:, 0], label='Configured bandpass')
plt.plot(np.arange(len(alternative_uV))/fs, alternative_uV[:, 0], alpha=.7, label='Exploratory 500 Hz lower cutoff')
plt.xlabel('s'); plt.ylabel('uV'); plt.legend()
